In [1]:

"""
Complete example demonstrating LiuEmbeddings framework.
"""

from liuembeddings import LiuEmbeddings
from liuembeddings import LiuVectorStore
from liuembeddings import split_text


c:\Users\himan\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Example 1: Basic embedding
print("=" * 60)
print("Example 1: Basic Embedding")
print("=" * 60)

print("\nInitializing embedding model and vector store...")

#embedder = LiuEmbeddings(model_name="BERT")
embedder = LiuEmbeddings(model_name="USE")

vector_store = LiuVectorStore(
    embedding_model=embedder,
    collection_name="ml_knowledge"
)

# Single query embedding
query_embedding = embedder.embed_query("What is machine learning?")
print(f"Query embedding dimension: {len(query_embedding)}")
print(f"First 5 values: {query_embedding[:5]}")

# Multiple documents
documents = [
    "Machine learning is a subset of AI",
    "Deep learning uses neural networks",
    "Natural language processing handles text"
]

doc_embeddings = embedder.embed_documents(documents)
print(f"\nEmbedded {len(doc_embeddings)} documents")


2025-10-31 01:01:13,044 - liuembeddings.embeddings - INFO - Loading model 'USE' (intfloat/e5-base-v2)


Example 1: Basic Embedding

Initializing embedding model and vector store...


2025-10-31 01:01:17,284 - liuembeddings.embeddings - INFO - ✅ Loaded 'USE' model (dim=768)
2025-10-31 01:01:17,287 - liuembeddings.vectorstore - INFO - Initializing ChromaDB at ./liu_db
2025-10-31 01:01:17,580 - liuembeddings.vectorstore - INFO - ✅ Initialized collection 'ml_knowledge' (documents: 0)
2025-10-31 01:01:17,791 - liuembeddings.embeddings - INFO - Embedded 3 documents


Query embedding dimension: 768
First 5 values: [-0.03044653870165348, -0.021079234778881073, -0.05617398023605347, -0.01297532394528389, 0.05516486242413521]

Embedded 3 documents


In [3]:
# Example 2: Text Processing
print("\n" + "=" * 60)
print("Example 2: Text Processing & Chunking")
print("=" * 60)

long_text = """
Machine learning is a powerful and rapidly growing method of data analysis that automates the process of building analytical models. It belongs to the broader field of artificial intelligence (AI), which focuses on creating systems that can simulate aspects of human intelligence. The central idea behind machine learning is that computers can be trained to learn from data, recognize patterns, and make informed decisions with little or no direct human intervention.

In traditional programming, humans explicitly write rules and logic for a computer to follow. However, in machine learning, these rules are not hard-coded. Instead, the system uses algorithms that allow it to learn and improve automatically from experience. The more data it receives, the better it becomes at identifying patterns, relationships, and insights that may not be obvious to humans. This ability to adapt and refine itself makes machine learning extremely valuable for handling complex and large-scale data problems.

The learning process begins with observations or data — these could come from examples, historical records, direct measurements, or real-world experiences. The model examines the data to detect recurring trends, correlations, or hidden patterns. Using this knowledge, it develops a mathematical representation that can be applied to new data to predict outcomes or make decisions. Over time, as more examples are processed, the model continuously updates and becomes more accurate.

Machine learning can be applied to an astonishing variety of real-world scenarios. For example, in healthcare, it helps predict diseases, recommend treatments, and analyze medical images. In finance, it detects fraudulent transactions and supports algorithmic trading. In e-commerce, machine learning powers recommendation engines that suggest products based on user behavior. Even in daily life, it appears in voice assistants like Siri or Alexa, which improve through constant interaction with users.

There are several types of machine learning approaches, including supervised learning, unsupervised learning, semi-supervised learning, and reinforcement learning. Supervised learning involves training a model with labeled data, where the correct output is known, while unsupervised learning deals with unlabeled data, letting the system find structure on its own. Reinforcement learning, on the other hand, trains models through trial and error, rewarding successful outcomes and penalizing mistakes. Each approach has unique strengths and is suited to different kinds of problems.

Ultimately, the goal of machine learning is to enable computers to make better and more autonomous decisions over time, based on the information they encounter. By continuously learning from new data, these systems can adjust their strategies, correct errors, and enhance performance without explicit reprogramming. This capability represents a major step toward the development of intelligent systems that can assist humans in solving complex challenges and making data-driven decisions with unprecedented speed and precision.

""" 


# Split into chunks
chunks = split_text(long_text, chunk_size=400, chunk_overlap=50)
print(f"\nSplit into {len(chunks)} chunks:")
for i, chunk in enumerate(chunks[:2], 1):
    print(f"  Chunk {i}: {chunk[:60]}...")


#add cleaned chunks to vector store
vector_store.add_texts(chunks)

# Query the vector store
print("\nQuerying the vector store...")
ans =vector_store.query("What techniques improve model accuracy?", n_results=2)

ans=ans[1]  #getting only documents from returned tuple


for i, chunk in enumerate(ans, 1):
    print(f"  answer {i}: {chunk[:250]}...")



2025-10-31 01:01:36,251 - liuembeddings.utils - INFO - Split text into 9 chunks (size: 400, overlap: 50)



Example 2: Text Processing & Chunking

Split into 9 chunks:
  Chunk 1: machine learning is a powerful and rapidly growing method of...
  Chunk 2: the central idea behind machine learning is that computers c...


2025-10-31 01:01:37,114 - liuembeddings.embeddings - INFO - Embedded 9 documents
2025-10-31 01:01:37,281 - liuembeddings.vectorstore - INFO - ✅ Added 9 documents to 'ml_knowledge'
2025-10-31 01:01:37,377 - liuembeddings.embeddings - INFO - Embedded 1 documents
2025-10-31 01:01:37,450 - liuembeddings.vectorstore - INFO - Query returned 2 results



Querying the vector store...
  answer 1: over time, as more examples are processed, the model continuously updates and becomes more accurate. machine learning can be applied to an astonishing variety of real-world scenarios. for example, in healthcare, it helps predict diseases, recommend t...
  answer 2: the learning process begins with observations or data — these could come from examples, historical records, direct measurements, or real-world experiences. the model examines the data to detect recurring trends, correlations, or hidden patterns. usin...


In [4]:
# Example 3: Vector Store Operations
print("\n" + "=" * 60)
print("Example 3: Vector Store CRUD Operations")
print("=" * 60)



# Add documents
docs_to_store = [
    "Machine learning is used for predictive analysis",
    "Deep learning requires large amounts of data",
    "Feature engineering is crucial for model performance"
]
vector_store.add_texts(docs_to_store)
print(f"Added {vector_store.count_documents()} documents")

# Search
raw,results = vector_store.similarity_search(
    "What techniques improve model accuracy?",
    n_results=1,
    with_score=.03
)

print(f"\nSearch results:")
for i, result in enumerate(results, 1):
    print(f"  Result {i} (score: {result['similarity_score']:.3f})")
    print(f"    {result['document'][:60]}...")

# Update document
print("\nUpdating first document...")
vector_store.update_by_id(
    results[0]['id'],
    "Machine learning drives innovation and efficiency"
)

# Search by ID
print(f"\nRetrieving by ID: {results[0]['id']}")
retrieved = vector_store.search_by_id(results[0]['id'])
print(f"  {retrieved['document']}")

# Get all documents
all_docs = vector_store.get_all()
print(f"\nTotal documents in store: {len(all_docs)}")


2025-10-31 01:02:00,930 - liuembeddings.embeddings - INFO - Embedded 3 documents
2025-10-31 01:02:00,949 - liuembeddings.vectorstore - INFO - ✅ Added 3 documents to 'ml_knowledge'
2025-10-31 01:02:01,010 - liuembeddings.embeddings - INFO - Embedded 1 documents



Example 3: Vector Store CRUD Operations
Added 12 documents

Search results:
  Result 1 (score: 0.810)
    over time, as more examples are processed, the model continu...

Updating first document...


2025-10-31 01:02:01,213 - liuembeddings.embeddings - INFO - Embedded 1 documents
2025-10-31 01:02:01,233 - liuembeddings.vectorstore - INFO - ✅ Updated document 'doc_4_1761852696253_bc8a62'
2025-10-31 01:02:01,241 - liuembeddings.vectorstore - INFO - Retrieved 12 total documents



Retrieving by ID: doc_4_1761852696253_bc8a62
  Machine learning drives innovation and efficiency

Total documents in store: 12


In [5]:
# Example 4: Batch Processing
print("\n" + "=" * 60)
print("Example 4: Batch Processing Large Documents")
print("=" * 60)

# Create many documents
large_doc_set = [f"Document number {i} with content about topic {i % 5}" 
                 for i in range(25)]

embedder_batch = LiuEmbeddings()
embeddings_batch = embedder_batch.embed_documents_batch(
    large_doc_set,
    batch_size=10
)
print(f"Processed {len(embeddings_batch)} documents in batches")


#batch storing (batch of 10 in each run 10 document will be added)
# 20 sample documents
texts = [f"Document {i+1}: This is sample text for document {i+1}." for i in range(100)]

# Optional metadata
metadatas = [{"source":"Batch of 5"} for i in range(100)]

vector_store.add_texts_batch(
    texts,
    batch_size=10,
    metadatas=metadatas
)

results=vector_store.search_by_metadata({"source":"Batch of 5"})

for i, result in enumerate(results, 1):
    print(f"   id: {result['id']}")
    print(f"    {result['document'][:60]}...")


2025-10-31 01:02:20,888 - liuembeddings.embeddings - INFO - Loading model 'MiniLM' (sentence-transformers/all-MiniLM-L6-v2)



Example 4: Batch Processing Large Documents


2025-10-31 01:02:25,194 - liuembeddings.embeddings - INFO - ✅ Loaded 'MiniLM' model (dim=384)
2025-10-31 01:02:25,195 - liuembeddings.embeddings - INFO - Processing 25 docs in 3 batches
2025-10-31 01:02:25,245 - liuembeddings.embeddings - INFO - Embedded 10 documents
2025-10-31 01:02:25,292 - liuembeddings.embeddings - INFO - Embedded 10 documents
2025-10-31 01:02:25,317 - liuembeddings.embeddings - INFO - Embedded 5 documents
2025-10-31 01:02:25,318 - liuembeddings.embeddings - INFO - ✅ Batch processing complete (25 documents)
2025-10-31 01:02:25,320 - liuembeddings.vectorstore - INFO - Adding 100 documents in 10 batches


Processed 25 documents in batches


2025-10-31 01:02:25,567 - liuembeddings.embeddings - INFO - Embedded 10 documents
2025-10-31 01:02:25,608 - liuembeddings.vectorstore - INFO - ✅ Added 10 documents to 'ml_knowledge'
2025-10-31 01:02:25,847 - liuembeddings.embeddings - INFO - Embedded 10 documents
2025-10-31 01:02:25,873 - liuembeddings.vectorstore - INFO - ✅ Added 10 documents to 'ml_knowledge'
2025-10-31 01:02:26,131 - liuembeddings.embeddings - INFO - Embedded 10 documents
2025-10-31 01:02:26,172 - liuembeddings.vectorstore - INFO - ✅ Added 10 documents to 'ml_knowledge'
2025-10-31 01:02:26,393 - liuembeddings.embeddings - INFO - Embedded 10 documents
2025-10-31 01:02:26,430 - liuembeddings.vectorstore - INFO - ✅ Added 10 documents to 'ml_knowledge'
2025-10-31 01:02:26,674 - liuembeddings.embeddings - INFO - Embedded 10 documents
2025-10-31 01:02:26,701 - liuembeddings.vectorstore - INFO - ✅ Added 10 documents to 'ml_knowledge'
2025-10-31 01:02:26,925 - liuembeddings.embeddings - INFO - Embedded 10 documents
2025-10-

   id: doc_0_1761852745321_a70b03
    Document 1: This is sample text for document 1....
   id: doc_1_1761852745321_791f2a
    Document 2: This is sample text for document 2....
   id: doc_2_1761852745321_31d794
    Document 3: This is sample text for document 3....
   id: doc_3_1761852745322_2a6fce
    Document 4: This is sample text for document 4....
   id: doc_4_1761852745322_19cb09
    Document 5: This is sample text for document 5....
   id: doc_5_1761852745322_ad3519
    Document 6: This is sample text for document 6....
   id: doc_6_1761852745322_e4493c
    Document 7: This is sample text for document 7....
   id: doc_7_1761852745322_37bfe5
    Document 8: This is sample text for document 8....
   id: doc_8_1761852745322_eb8dca
    Document 9: This is sample text for document 9....
   id: doc_9_1761852745322_737b55
    Document 10: This is sample text for document 10....
   id: doc_0_1761852745609_322a7f
    Document 11: This is sample text for document 11....
   id: doc_1_1761

In [6]:
# Example 5: One-Line Search one function ultimate usage
print("\n" + "=" * 60)
print("Example 5: One-Line Semantic Search")
print("=" * 60)

print("\nPerforming semantic search using liu_search... \n all in embedding solution")

document = """
Python is a high-level programming language known for its simplicity.
JavaScript is used primarily for web development.
Java is popular for enterprise applications.
C++ is known for high performance and system programming.
""" * 2

#try to always  pass text in text_document parameter while adding 

#adding and searching
raw,ans = vector_store.search(
    "What language is best for web development?",
    text_document=document,
    n_results=1
)

for i in range(len(ans)):
    print(f"  Result {i+1}: {ans[i][:90]}...")




print("\nPerforming another semantic search using search... \n only adding document")
document = """
my name is himanshu i am a data engineer working in tcs india pvt ltd.
i have experience in spark,hadoop,python,sql,azure,aws,tableau,power bi etc.
i love to work on data and build data pipelines and dashboards.
python is a high-level programming language known for its simplicity but it is not simple :).
""" 


#adding only
vector_store.search(
    text_document=document,
    chunk_size=250,
    chunk_overlap=100,
)


print("\nPerforming another semantic search using liu_search... \n only searching")
#searching only
search_results = vector_store.search(
    query="What himanshu does for a living?",
    n_results=1
)

2025-10-31 01:02:30,423 - liuembeddings.vectorstore - INFO - Starting liu_search operation
2025-10-31 01:02:30,424 - liuembeddings.vectorstore - INFO - Splitting text (size: 1000, overlap: 200)
2025-10-31 01:02:30,425 - liuembeddings.utils - INFO - Split text into 1 chunks (size: 1000, overlap: 200)
2025-10-31 01:02:30,426 - liuembeddings.vectorstore - INFO - Adding 1 chunks to vector store
2025-10-31 01:02:30,596 - liuembeddings.embeddings - INFO - Embedded 1 documents



Example 5: One-Line Semantic Search

Performing semantic search using liu_search... 
 all in embedding solution


2025-10-31 01:02:30,625 - liuembeddings.vectorstore - INFO - ✅ Added 1 documents to 'ml_knowledge'
2025-10-31 01:02:30,626 - liuembeddings.vectorstore - INFO - Searching for: 'What language is best for web development?'
2025-10-31 01:02:30,705 - liuembeddings.embeddings - INFO - Embedded 1 documents
2025-10-31 01:02:30,708 - liuembeddings.vectorstore - INFO - Query returned 1 results
2025-10-31 01:02:30,709 - liuembeddings.vectorstore - INFO - ✅ Search complete. Found 2 results
2025-10-31 01:02:30,709 - liuembeddings.vectorstore - INFO - Starting liu_search operation
2025-10-31 01:02:30,710 - liuembeddings.vectorstore - INFO - Splitting text (size: 250, overlap: 100)
2025-10-31 01:02:30,711 - liuembeddings.utils - INFO - Split text into 2 chunks (size: 250, overlap: 100)
2025-10-31 01:02:30,711 - liuembeddings.vectorstore - INFO - Adding 2 chunks to vector store


  Result 1: python is a high-level programming language known for its simplicity. javascript is used p...

Performing another semantic search using search... 
 only adding document


2025-10-31 01:02:30,923 - liuembeddings.embeddings - INFO - Embedded 2 documents
2025-10-31 01:02:30,958 - liuembeddings.vectorstore - INFO - ✅ Added 2 documents to 'ml_knowledge'
2025-10-31 01:02:30,959 - liuembeddings.vectorstore - INFO - Searching for: 'What himanshu does for a living?'
2025-10-31 01:02:31,042 - liuembeddings.embeddings - INFO - Embedded 1 documents
2025-10-31 01:02:31,046 - liuembeddings.vectorstore - INFO - Query returned 1 results
2025-10-31 01:02:31,048 - liuembeddings.vectorstore - INFO - ✅ Search complete. Found 2 results



Performing another semantic search using liu_search... 
 only searching


In [7]:
from liuembeddings import fastquery


#when with score is false(deafult is false) it returns raw,document where doucument is list of matching documents
document = """
Luna loves exploring the night sky. Every weekend, she sets up her telescope on the rooftop to watch distant galaxies.
Her favorite constellation is Orion, and she can identify it even without a telescope.
Last month, she discovered a small comet passing near Jupiter and recorded its movement in her astronomy journal.
"""
fastquery(
    text_document=document,
    n_results=5,       
    collection_name="story_collection", 
)



raw,ans = fastquery(
    query="What celestial object did Luna discover?",
    collection_name="story_collection", 
)


for item in ans:
    print(f"Answer: {item}")


#when with score is true it reurns a raw,list where list is  dict id,document,metadata,similarity score


raw,ans = fastquery(
    query="What celestial object did Luna discover?",
    with_score=0.5,   #
    collection_name="story_collection"
)


for item in ans:
    print(f"id: {item['id']}")
    print(f"Docs: {item['document']}")
    print(f"Metadeta: {item['metadata']}")
    print(f"Similarity score: {item['similarity_score']}")


2025-10-31 01:02:39,193 - liuembeddings.liu_search - INFO - Loading embedding model: USE
2025-10-31 01:02:39,195 - liuembeddings.embeddings - INFO - ✅ Loaded 'USE' model (dim=768)
2025-10-31 01:02:39,196 - liuembeddings.liu_search - INFO - Creating vector store: story_collection
2025-10-31 01:02:39,197 - liuembeddings.vectorstore - INFO - Initializing ChromaDB at ./liu_db
2025-10-31 01:02:39,336 - liuembeddings.vectorstore - INFO - ✅ Initialized collection 'story_collection' (documents: 0)
2025-10-31 01:02:39,337 - liuembeddings.liu_search - INFO - Starting liu_search operation
2025-10-31 01:02:39,339 - liuembeddings.liu_search - INFO - Splitting text (size: 1000, overlap: 200)
2025-10-31 01:02:39,339 - liuembeddings.utils - INFO - Split text into 1 chunks (size: 1000, overlap: 200)
2025-10-31 01:02:39,340 - liuembeddings.liu_search - INFO - Adding 1 chunks to vector store
2025-10-31 01:02:39,483 - liuembeddings.embeddings - INFO - Embedded 1 documents
2025-10-31 01:02:39,644 - liuembe

Answer: luna loves exploring the night sky. every weekend, she sets up her telescope on the rooftop to watch distant galaxies. her favorite constellation is orion, and she can identify it even without a telescope. last month, she discovered a small comet passing near jupiter and recorded its movement in her astronomy journal.


2025-10-31 01:02:40,111 - liuembeddings.embeddings - INFO - Embedded 1 documents
2025-10-31 01:02:40,113 - liuembeddings.liu_search - INFO - ✅ Search complete. Found 2 results


id: doc_0_1761852759340_e5b3ad
Docs: luna loves exploring the night sky. every weekend, she sets up her telescope on the rooftop to watch distant galaxies. her favorite constellation is orion, and she can identify it even without a telescope. last month, she discovered a small comet passing near jupiter and recorded its movement in her astronomy journal.
Metadeta: {'source': 'story_collection'}
Similarity score: 0.8489370346069336


In [8]:
# Example 6: Error Handling
print("\n" + "=" * 60)
print("Example 6: Error Handling")
print("=" * 60)

try:
    # This will raise an error
    embedder.embed_query("")
except ValueError as e:
    print(f"✓ Caught error: {e}")

try:
    # Invalid model
    LiuEmbeddings(model_name="INVALID")
except ValueError as e:
    print(f"✓ Caught error: {e}")

try:
    # Empty documents list
    vector_store.add_texts([])
except ValueError as e:
    print(f"✓ Caught error: {e}")


print("\n" + "=" * 60)
print("Examples completed successfully! ✅")
print("=" * 60)



Example 6: Error Handling
✓ Caught error: Text cannot be empty
✓ Caught error: Model 'INVALID' not found. Available: MiniLM, MPNetBase, USE, USEL
✓ Caught error: texts list cannot be empty

Examples completed successfully! ✅
